# Lesson 3c — Attention from scratch (runnable)

The most important "no black boxes" lesson. We implement multi-head self-attention from primitives — no `nn.MultiheadAttention`, no `TransformerEncoderLayer` — and verify it gives the same numerical result as PyTorch's built-in.

Runnable version of [`03c_attention_from_scratch.py`](../03c_attention_from_scratch.py).


## Imports

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
torch.set_printoptions(precision=3, sci_mode=False)

## Setup: batch of sequences

Shape convention: `(B, L, d_model)` — batch, sequence length, embedding dim.

In [ ]:
B, L, d_model = 2, 4, 6
n_heads = 2
d_k = d_model // n_heads                    # Dimension per head.

print(f"Batch B={B}, length L={L}, d_model={d_model}, n_heads={n_heads}, d_k={d_k}")
x = torch.randn(B, L, d_model)
print(f"x.shape = {tuple(x.shape)}")

## Single-head attention, step by step

The formula: `Attention(Q, K, V) = softmax(Q @ K^T / sqrt(d_k)) @ V`.

In [ ]:
W_q = nn.Linear(d_model, d_model, bias=False)
W_k = nn.Linear(d_model, d_model, bias=False)
W_v = nn.Linear(d_model, d_model, bias=False)
W_o = nn.Linear(d_model, d_model, bias=False)

Q = W_q(x)                                  # 'What am I looking for?'
K = W_k(x)                                  # 'What do I offer?'
V = W_v(x)                                  # 'What do I contribute if matched?'

print(f"Q.shape = {tuple(Q.shape)}")
print(f"K.shape = {tuple(K.shape)}")
print(f"V.shape = {tuple(V.shape)}")

scores = Q @ K.transpose(-2, -1)            # Pairwise dot products. (B, L, L).
scores = scores / math.sqrt(d_model)        # Scale to keep softmax sharp but not too sharp.
attn = F.softmax(scores, dim=-1)            # Each row sums to 1.
out = attn @ V                              # Weighted sum of values.
out = W_o(out)                              # Final output projection.

print(f"\nout.shape = {tuple(out.shape)}")

## Multi-head attention from scratch

Run `n_heads` attentions in parallel with different Q/K/V slices, then concatenate.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        B, L, _ = x.shape
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)
        # Split into heads. (B, L, d) -> (B, n_heads, L, d_k)
        Q = Q.view(B, L, self.n_heads, self.d_k).transpose(1, 2)
        K = K.view(B, L, self.n_heads, self.d_k).transpose(1, 2)
        V = V.view(B, L, self.n_heads, self.d_k).transpose(1, 2)
        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_k)
        attn = F.softmax(scores, dim=-1)
        out = attn @ V
        # Concat heads
        out = out.transpose(1, 2).contiguous().view(B, L, self.d_model)
        return self.W_o(out)

mha = MultiHeadAttention(d_model, n_heads)
out_mine = mha(x)
print(f"Multi-head output shape: {tuple(out_mine.shape)}")

## Sanity check vs `nn.MultiheadAttention`

Copy our weights into PyTorch's version and verify identical output.

In [ ]:
ref = nn.MultiheadAttention(d_model, n_heads, bias=False, batch_first=True)

with torch.no_grad():
    # PyTorch concatenates Q/K/V into one in_proj_weight
    ref.in_proj_weight.copy_(torch.cat([mha.W_q.weight, mha.W_k.weight, mha.W_v.weight], dim=0))
    ref.out_proj.weight.copy_(mha.W_o.weight)

out_ref, _ = ref(x, x, x, need_weights=False)

diff = (out_mine - out_ref).abs().max().item()
print(f"Max absolute difference: {diff:.2e}")
if diff < 1e-5:
    print("✓ Numerically identical. Our 25-line MultiHeadAttention IS nn.MultiheadAttention.")

## Things to try

1. Drop the `sqrt(d_k)` scaling. What happens to training?
2. Set `n_heads = 1` — verify it matches single-head attention.
3. Add a causal MASK (an L×L matrix of `-inf` on the future positions). This is how GPT works.
4. Print the attention matrix during training — does it learn to attend to specific tokens?